In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pytorch_lightning as pl
import pandas as pd
from time import time

In [2]:

# =====================================================
# 1. Dataset Class
# =====================================================
class OSV5MDataset(Dataset):
    """Dataset for OSV-5M with flat directory structure"""
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['id']}.jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        target = torch.tensor([row['latitude'], row['longitude']], dtype=torch.float32)
        return image, target


# =====================================================
# 2. Lightning DataModule
# =====================================================
class OSV5MModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, train_image_dir, test_image_dir, batch_size=32):
        super().__init__()
        self.train_df = train_df
        self.test_df = test_df
        self.train_image_dir = train_image_dir
        self.test_image_dir = test_image_dir
        self.batch_size = batch_size

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        self.train_dataset = OSV5MDataset(self.train_df, self.train_image_dir, self.transform)
        self.test_dataset = OSV5MDataset(self.test_df, self.test_image_dir, self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)


# =====================================================
# 3. Lightning Model
# =====================================================
class ResNetRegressorPLM(pl.LightningModule):
    def __init__(self, learning_rate=1e-4):
        super().__init__()
        self.learning_rate = learning_rate
        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)
        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.criterion(outputs, targets)
        rmse = torch.sqrt(loss)
        self.log('train_loss', loss)
        self.log('train_rmse', rmse)
        return loss

    def test_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.criterion(outputs, targets)
        rmse = torch.sqrt(loss)
        self.log('test_loss', loss)
        self.log('test_rmse', rmse)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        return optimizer


In [ ]:
device_type = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {device_type.upper()} for training")

# Load CSVs directly
train_df = pd.read_csv("osv-5m/train.csv")
test_df = pd.read_csv("osv-5m/test.csv")

# Initialize DataModule and Model
geo_module = OSV5MModule(
    train_df, 
    test_df, 
    train_image_dir="osv-5m/images/train_flat", 
    test_image_dir="osv-5m/images/test_flat",
    batch_size=8)  # smaller batch size
geo_model = ResNetRegressorPLM(learning_rate=1e-4)

# Lightning Trainer
trainer = pl.Trainer(
    max_epochs=3,
    accelerator=device_type,  # 'gpu' or 'cpu'
    devices=1,
    log_every_n_steps=10
)

# Train + test
time_start = time()
trainer.fit(geo_model, datamodule=geo_module)
time_stop = time()

print(f"Training time: {round(time_stop - time_start, 1)} sec")

trainer.test(geo_model, datamodule=geo_module)

Using GPU for training


/tmp/ipykernel_1098173/2131102584.py:5: DtypeWarning: Columns (11,27) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("osv-5m/train.csv")
/work/cssema416/202610/28/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/work/cssema416/202610/28/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, wh

Epoch 0: 100%|█████████▉| 611020/611836 [4:00:59<00:19, 42.26it/s, v_num=16]  